In [1]:
import pandas as pd
import mysql.connector

In [2]:
conn=mysql.connector.connect(
    host="localhost",
    user="root",
    password="banumathi@#$0",
    database="revenue_prediction")
print('conect success')

conect success


In [11]:
query="""
WITH salesperson_leakage AS (
    SELECT
        so.salesperson_id,
        sp.salesperson_name,
        sp.region,

        COUNT(
            CASE
                WHEN (so.list_price * (1 - dp.max_discount_pct / 100)) - i.invoice_price > 0
                THEN so.order_id
            END
        ) AS leakage_orders,

        ROUND(
            SUM(
                CASE
                    WHEN (so.list_price * (1 - dp.max_discount_pct / 100)) - i.invoice_price > 0
                    THEN (so.list_price * (1 - dp.max_discount_pct / 100)) - i.invoice_price
                    ELSE 0
                END
            ),
            2
        ) AS total_leakage

    FROM sales_orders so
    JOIN invoices i
        ON so.order_id = i.order_id
    JOIN salespersons sp
        ON so.salesperson_id = sp.salesperson_id
    JOIN discount_policy dp
        ON so.product = dp.product

    GROUP BY
        so.salesperson_id,
        sp.salesperson_name,
        sp.region
)

SELECT
    *,
    RANK() OVER (ORDER BY total_leakage DESC) AS risk_rank
FROM salesperson_leakage
WHERE total_leakage > 0
ORDER BY risk_rank;
"""

In [12]:
df = pd.read_sql(query, conn)
df.head()


C:\Users\ELCOT\AppData\Local\Temp\ipykernel_16144\3140185810.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,salesperson_id,salesperson_name,region,leakage_orders,total_leakage,risk_rank
0,SP013,Salesperson_13,North,37,536652.73,1
1,SP005,Salesperson_5,South,26,502882.44,2
2,SP004,Salesperson_4,East,26,460060.15,3
3,SP006,Salesperson_6,East,24,436880.55,4
4,SP003,Salesperson_3,South,24,410323.96,5


In [15]:
df.to_csv("salesperson_risk_summary.csv", index=False)
